# Complete ML Market Prediction System - All-in-One Notebook

This notebook contains all ML modules in one place. You can run everything from here!

## Contents:
1. Feature Engineering Module
2. ML Models Module  
3. ML Trainer Module
4. ML Predictor Module
5. ML Engine Module
6. Usage Examples

---
# 1. Feature Engineering Module
Extract 50+ technical indicators from market data

In [ ]:
# Install dependencies first (run once)
# !pip install yfinance pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Optional, Tuple
from datetime import datetime, timedelta
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

class FeatureEngineer:
    """
    Comprehensive feature extraction for market prediction.
    Generates 50+ features from price/volume data.
    """
    
    def __init__(self):
        self.feature_names = []
        
    def calculate_rsi(self, prices: pd.Series, period: int = 14) -> pd.Series:
        """Calculate Relative Strength Index"""
        delta = prices.diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        return rsi
    
    def calculate_macd(self, prices: pd.Series, fast: int = 12, slow: int = 26, signal: int = 9) -> Tuple[pd.Series, pd.Series, pd.Series]:
        """Calculate MACD, Signal line, and Histogram"""
        ema_fast = prices.ewm(span=fast, adjust=False).mean()
        ema_slow = prices.ewm(span=slow, adjust=False).mean()
        macd = ema_fast - ema_slow
        signal_line = macd.ewm(span=signal, adjust=False).mean()
        histogram = macd - signal_line
        return macd, signal_line, histogram
    
    def calculate_bollinger_bands(self, prices: pd.Series, period: int = 20, std_dev: float = 2.0) -> Tuple[pd.Series, pd.Series, pd.Series]:
        """Calculate Bollinger Bands"""
        sma = prices.rolling(window=period).mean()
        std = prices.rolling(window=period).std()
        upper_band = sma + (std * std_dev)
        lower_band = sma - (std * std_dev)
        return upper_band, sma, lower_band
    
    def calculate_atr(self, high: pd.Series, low: pd.Series, close: pd.Series, period: int = 14) -> pd.Series:
        """Calculate Average True Range"""
        tr1 = high - low
        tr2 = abs(high - close.shift())
        tr3 = abs(low - close.shift())
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = tr.rolling(window=period).mean()
        return atr
    
    def extract_features(self, symbol: str, lookback_days: int = 365) -> pd.DataFrame:
        """Extract all features for a given symbol"""
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)
        
        data = yf.download(symbol, start=start_date, end=end_date, progress=False)
        
        if data.empty:
            raise ValueError(f"No data available for {symbol}")
        
        df = pd.DataFrame()
        
        # Price features
        df['close'] = data['Close']
        df['open'] = data['Open']
        df['high'] = data['High']
        df['low'] = data['Low']
        df['volume'] = data['Volume']
        
        # Returns
        df['return_1d'] = df['close'].pct_change()
        df['return_5d'] = df['close'].pct_change(5)
        df['return_20d'] = df['close'].pct_change(20)
        
        # Moving Averages
        for period in [5, 10, 20, 50, 200]:
            df[f'sma_{period}'] = df['close'].rolling(window=period).mean()
            df[f'ema_{period}'] = df['close'].ewm(span=period, adjust=False).mean()
            df[f'price_to_sma_{period}'] = df['close'] / df[f'sma_{period}']
        
        # Technical Indicators
        df['rsi_14'] = self.calculate_rsi(df['close'], 14)
        df['rsi_7'] = self.calculate_rsi(df['close'], 7)
        
        macd, signal, histogram = self.calculate_macd(df['close'])
        df['macd'] = macd
        df['macd_signal'] = signal
        df['macd_histogram'] = histogram
        
        upper_bb, middle_bb, lower_bb = self.calculate_bollinger_bands(df['close'])
        df['bb_upper'] = upper_bb
        df['bb_middle'] = middle_bb
        df['bb_lower'] = lower_bb
        df['bb_width'] = (upper_bb - lower_bb) / middle_bb
        df['bb_position'] = (df['close'] - lower_bb) / (upper_bb - lower_bb)
        
        df['atr_14'] = self.calculate_atr(df['high'], df['low'], df['close'], 14)
        
        # Volatility
        df['volatility_20'] = df['return_1d'].rolling(window=20).std()
        
        # Momentum
        df['momentum_5'] = df['close'] - df['close'].shift(5)
        df['momentum_10'] = df['close'] - df['close'].shift(10)
        
        # Target Variable
        df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
        
        df = df.dropna()
        
        self.feature_names = [col for col in df.columns if col not in ['target', 'close', 'open', 'high', 'low', 'volume']]
        
        return df
    
    def prepare_training_data(self, symbol: str, lookback_days: int = 730) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare features (X) and target (y) for model training"""
        df = self.extract_features(symbol, lookback_days)
        X = df[self.feature_names]
        y = df['target']
        return X, y

print("✅ FeatureEngineer class loaded")

---
# 2. ML Models Module
Ensemble models (Random Forest, XGBoost, LightGBM) with stacking

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import os

class EnsembleModel:
    """
    Ensemble of multiple ML models for robust predictions.
    Combines Random Forest, XGBoost, and LightGBM.
    """
    
    def __init__(self, model_dir: str = "ml_models"):
        self.model_dir = model_dir
        os.makedirs(model_dir, exist_ok=True)
        
        self.models = {
            'random_forest': RandomForestClassifier(
                n_estimators=200, max_depth=15, min_samples_split=10,
                random_state=42, n_jobs=-1
            ),
            'xgboost': XGBClassifier(
                n_estimators=200, max_depth=8, learning_rate=0.05,
                random_state=42, n_jobs=-1, eval_metric='logloss'
            ),
            'lightgbm': LGBMClassifier(
                n_estimators=200, max_depth=8, learning_rate=0.05,
                random_state=42, n_jobs=-1, verbose=-1
            )
        }
        
        self.meta_learner = LogisticRegression(random_state=42, max_iter=1000)
        self.is_trained = False
        self.feature_importance = {}
    
    def train(self, X_train: pd.DataFrame, y_train: pd.Series, 
              X_val: Optional[pd.DataFrame] = None, y_val: Optional[pd.Series] = None):
        """Train all ensemble models"""
        print("Training ensemble models...")
        
        for name, model in self.models.items():
            print(f"  Training {name}...")
            model.fit(X_train, y_train)
            
            if hasattr(model, 'feature_importances_'):
                self.feature_importance[name] = dict(zip(
                    X_train.columns, model.feature_importances_
                ))
            
            if X_val is not None and y_val is not None:
                val_score = model.score(X_val, y_val)
                print(f"    {name} validation accuracy: {val_score:.4f}")
        
        # Train meta-learner
        print("  Training meta-learner...")
        meta_features_train = self._get_meta_features(X_train)
        self.meta_learner.fit(meta_features_train, y_train)
        
        self.is_trained = True
        print("Ensemble training complete!")
    
    def _get_meta_features(self, X: pd.DataFrame) -> np.ndarray:
        """Get predictions from base models as meta-features"""
        meta_features = []
        for model in self.models.values():
            probs = model.predict_proba(X)[:, 1]
            meta_features.append(probs)
        return np.column_stack(meta_features)
    
    def predict(self, X: pd.DataFrame, use_stacking: bool = True) -> np.ndarray:
        """Make predictions"""
        if not self.is_trained:
            raise ValueError("Models not trained yet!")
        
        if use_stacking:
            meta_features = self._get_meta_features(X)
            return self.meta_learner.predict(meta_features)
        else:
            predictions = [model.predict(X) for model in self.models.values()]
            return np.round(np.mean(predictions, axis=0)).astype(int)
    
    def predict_proba(self, X: pd.DataFrame, use_stacking: bool = True) -> np.ndarray:
        """Get probability predictions"""
        if not self.is_trained:
            raise ValueError("Models not trained yet!")
        
        if use_stacking:
            meta_features = self._get_meta_features(X)
            return self.meta_learner.predict_proba(meta_features)
        else:
            all_probs = [model.predict_proba(X) for model in self.models.values()]
            return np.mean(all_probs, axis=0)
    
    def get_confidence_score(self, X: pd.DataFrame) -> float:
        """Get confidence score based on model agreement"""
        predictions = [model.predict(X) for model in self.models.values()]
        predictions = np.array(predictions)
        agreement = np.mean(predictions == predictions[0])
        return float(agreement)
    
    def save(self, symbol: str, version: str = "v1"):
        """Save all models to disk"""
        model_path = os.path.join(self.model_dir, f"{symbol}_{version}")
        os.makedirs(model_path, exist_ok=True)
        
        for name, model in self.models.items():
            joblib.dump(model, os.path.join(model_path, f"{name}.pkl"))
        
        joblib.dump(self.meta_learner, os.path.join(model_path, "meta_learner.pkl"))
        joblib.dump(self.feature_importance, os.path.join(model_path, "feature_importance.pkl"))
        
        print(f"Models saved to {model_path}")
    
    def load(self, symbol: str, version: str = "v1"):
        """Load models from disk"""
        model_path = os.path.join(self.model_dir, f"{symbol}_{version}")
        
        if not os.path.exists(model_path):
            raise FileNotFoundError(f"Model path {model_path} not found")
        
        for name in self.models.keys():
            self.models[name] = joblib.load(os.path.join(model_path, f"{name}.pkl"))
        
        self.meta_learner = joblib.load(os.path.join(model_path, "meta_learner.pkl"))
        self.feature_importance = joblib.load(os.path.join(model_path, "feature_importance.pkl"))
        
        self.is_trained = True
        print(f"Models loaded from {model_path}")
    
    def get_top_features(self, top_n: int = 10) -> Dict[str, List[Tuple[str, float]]]:
        """Get top N most important features for each model"""
        top_features = {}
        
        for model_name, importance_dict in self.feature_importance.items():
            sorted_features = sorted(
                importance_dict.items(), 
                key=lambda x: x[1], 
                reverse=True
            )[:top_n]
            top_features[model_name] = sorted_features
        
        return top_features

print("✅ EnsembleModel class loaded")

---
# 3. ML Trainer Module
Training pipeline with metrics and evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

class MLTrainer:
    """Complete training pipeline for market prediction models"""
    
    def __init__(self, symbol: str, lookback_days: int = 730):
        self.symbol = symbol
        self.lookback_days = lookback_days
        self.engineer = FeatureEngineer()
        self.ensemble_model = EnsembleModel()
        self.metrics = {}
        
    def prepare_data(self) -> Tuple[pd.DataFrame, pd.Series]:
        """Prepare training data"""
        print(f"Preparing data for {self.symbol}...")
        X, y = self.engineer.prepare_training_data(self.symbol, self.lookback_days)
        print(f"Data shape: {X.shape}")
        print(f"Target distribution: UP={sum(y)}, DOWN={len(y)-sum(y)}")
        return X, y
    
    def train_ensemble(self, X: pd.DataFrame, y: pd.Series, test_size: float = 0.2):
        """Train ensemble model with train/test split"""
        split_idx = int(len(X) * (1 - test_size))
        
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]
        
        print(f"\nTraining ensemble on {len(X_train)} samples...")
        print(f"Testing on {len(X_test)} samples...")
        
        self.ensemble_model.train(X_train, y_train, X_test, y_test)
        
        # Evaluate
        y_pred = self.ensemble_model.predict(X_test, use_stacking=True)
        y_proba = self.ensemble_model.predict_proba(X_test, use_stacking=True)[:, 1]
        
        self.metrics['ensemble'] = self._calculate_metrics(y_test, y_pred, y_proba)
        
        print("\n=== Ensemble Model Performance ===")
        self._print_metrics(self.metrics['ensemble'])
        
        return X_train, X_test, y_train, y_test
    
    def _calculate_metrics(self, y_true, y_pred, y_proba) -> Dict:
        """Calculate evaluation metrics"""
        metrics = {
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
        }
        
        try:
            metrics['roc_auc'] = roc_auc_score(y_true, y_proba)
        except:
            metrics['roc_auc'] = 0.0
        
        return metrics
    
    def _print_metrics(self, metrics: Dict):
        """Print metrics in a formatted way"""
        print(f"  Accuracy:  {metrics['accuracy']:.4f}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall:    {metrics['recall']:.4f}")
        print(f"  F1 Score:  {metrics['f1']:.4f}")
        print(f"  ROC-AUC:   {metrics['roc_auc']:.4f}")
    
    def get_feature_importance(self, top_n: int = 15):
        """Get and display top features"""
        top_features = self.ensemble_model.get_top_features(top_n)
        
        print(f"\n=== Top {top_n} Most Important Features ===")
        for model_name, features in top_features.items():
            print(f"\n{model_name.upper()}:")
            for i, (feature, importance) in enumerate(features, 1):
                print(f"  {i}. {feature}: {importance:.4f}")
        
        return top_features
    
    def save_models(self, version: str = "v1"):
        """Save trained models"""
        print(f"\nSaving models for {self.symbol} (version: {version})...")
        self.ensemble_model.save(self.symbol, version)
        print("Models saved successfully!")
    
    def full_training_pipeline(self, test_size: float = 0.2, save_models: bool = True):
        """Run complete training pipeline"""
        print(f"\n{'='*60}")
        print(f"TRAINING PIPELINE FOR {self.symbol}")
        print(f"{'='*60}")
        
        X, y = self.prepare_data()
        X_train, X_test, y_train, y_test = self.train_ensemble(X, y, test_size)
        self.get_feature_importance()
        
        if save_models:
            self.save_models()
        
        print(f"\n{'='*60}")
        print(f"TRAINING COMPLETE!")
        print(f"{'='*60}")
        
        return self.metrics

print("✅ MLTrainer class loaded")

---
# 4. ML Predictor Module
Real-time prediction engine

In [ ]:
class MLPredictor:
    """Real-time prediction engine for market direction"""
    
    def __init__(self, symbol: str, model_version: str = "v1"):
        self.symbol = symbol
        self.model_version = model_version
        self.engineer = FeatureEngineer()
        self.ensemble_model = EnsembleModel()
        self._load_models()
    
    def _load_models(self):
        """Load trained models"""
        try:
            print(f"Loading models for {self.symbol} (version: {self.model_version})...")
            self.ensemble_model.load(self.symbol, self.model_version)
            print("✓ Ensemble model loaded")
        except FileNotFoundError:
            print(f"⚠ No trained ensemble model found for {self.symbol}")
            print("  Please train the model first")
            raise
    
    def predict(self, use_ensemble: bool = True) -> Dict:
        """Make prediction for the next trading session"""
        print(f"\nGenerating prediction for {self.symbol}...")
        
        try:
            X, y = self.engineer.prepare_training_data(self.symbol, lookback_days=365)
            X_latest = X.iloc[[-1]]
        except Exception as e:
            return {
                'error': f"Failed to fetch data: {str(e)}",
                'symbol': self.symbol,
                'timestamp': datetime.now().isoformat()
            }
        
        result = {
            'symbol': self.symbol,
            'timestamp': datetime.now().isoformat(),
            'predictions': {},
            'confidence': {},
            'final_prediction': None,
            'final_confidence': 0.0
        }
        
        if use_ensemble and self.ensemble_model.is_trained:
            ensemble_pred = self.ensemble_model.predict(X_latest, use_stacking=True)[0]
            ensemble_proba = self.ensemble_model.predict_proba(X_latest, use_stacking=True)[0]
            ensemble_confidence = self.ensemble_model.get_confidence_score(X_latest)
            
            result['predictions']['ensemble'] = 'UP' if ensemble_pred == 1 else 'DOWN'
            result['confidence']['ensemble'] = float(ensemble_confidence)
            result['probability'] = {
                'down': float(ensemble_proba[0]),
                'up': float(ensemble_proba[1])
            }
            
            result['final_prediction'] = result['predictions']['ensemble']
            result['final_confidence'] = result['confidence']['ensemble']
        
        result['risk_level'] = self._assess_risk(result['final_confidence'])
        
        return result
    
    def _assess_risk(self, confidence: float) -> str:
        """Assess risk level based on confidence score"""
        if confidence >= 0.8:
            return "LOW"
        elif confidence >= 0.6:
            return "MEDIUM"
        else:
            return "HIGH"
    
    def predict_with_explanation(self) -> Dict:
        """Make prediction with detailed explanation"""
        prediction = self.predict(use_ensemble=True)
        
        if 'error' in prediction:
            return prediction
        
        # Add interpretation
        direction = prediction['final_prediction']
        confidence = prediction['final_confidence']
        risk = prediction['risk_level']
        
        interpretation = f"The model predicts the market will go {direction} "
        interpretation += f"with {confidence*100:.1f}% confidence ({risk} risk)."
        
        prediction['interpretation'] = interpretation
        
        return prediction

def print_prediction(prediction: Dict):
    """Pretty print prediction results"""
    print("\n" + "="*60)
    print(f"MARKET PREDICTION: {prediction['symbol']}")
    print("="*60)
    
    if 'error' in prediction:
        print(f"❌ Error: {prediction['error']}")
        return
    
    print(f"\n📊 Final Prediction: {prediction['final_prediction']}")
    print(f"🎯 Confidence: {prediction['final_confidence']*100:.1f}%")
    print(f"⚠️  Risk Level: {prediction['risk_level']}")
    
    if 'probability' in prediction:
        prob = prediction['probability']
        print(f"\n📈 Probabilities:")
        print(f"   UP:   {prob['up']*100:.1f}%")
        print(f"   DOWN: {prob['down']*100:.1f}%")
    
    if 'interpretation' in prediction:
        print(f"\n💡 Interpretation:")
        print(f"   {prediction['interpretation']}")
    
    print("\n" + "="*60)

print("✅ MLPredictor class loaded")

---
# 5. Usage Examples
Now let's use everything we've built!

## Example 1: Train a Model

In [ ]:
# Train model for SPY (S&P 500)
symbol = 'SPY'

trainer = MLTrainer(symbol, lookback_days=730)
metrics = trainer.full_training_pipeline(test_size=0.2, save_models=True)

print(f"\n✅ Training complete for {symbol}!")
print(f"Final Accuracy: {metrics['ensemble']['accuracy']:.2%}")

## Example 2: Make a Prediction

In [ ]:
# Get prediction for SPY
predictor = MLPredictor('SPY', model_version='v1')
prediction = predictor.predict_with_explanation()

print_prediction(prediction)

## Example 3: Batch Predictions

In [ ]:
# Predict for multiple symbols
symbols = ['SPY', 'QQQ', 'DIA']
results = []

for sym in symbols:
    try:
        pred = MLPredictor(sym).predict()
        if 'error' not in pred:
            results.append({
                'Symbol': sym,
                'Direction': pred['final_prediction'],
                'Confidence': f"{pred['final_confidence']:.2%}",
                'Risk': pred['risk_level']
            })
    except:
        pass

if results:
    df = pd.DataFrame(results)
    display(df)

## Example 4: Visualize Feature Importance

In [ ]:
# Get feature importance from trained model
top_features = trainer.ensemble_model.get_top_features(top_n=10)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
model_names = ['random_forest', 'xgboost', 'lightgbm']

for idx, model_name in enumerate(model_names):
    if model_name in top_features:
        features = top_features[model_name]
        names = [f[0] for f in features]
        importances = [f[1] for f in features]
        
        axes[idx].barh(names, importances, color='steelblue')
        axes[idx].set_xlabel('Importance', fontweight='bold')
        axes[idx].set_title(f'{model_name.upper()}', fontweight='bold')
        axes[idx].invert_yaxis()

plt.tight_layout()
plt.show()

---
# 🎉 Complete!

You now have a fully functional ML market prediction system in one notebook!

### What you can do:
- ✅ Train models for any symbol
- ✅ Get real-time predictions
- ✅ Analyze feature importance
- ✅ Batch predictions for multiple symbols
- ✅ Visualize results

### Next steps:
- Train models for more symbols (AAPL, GOOGL, TSLA, etc.)
- Experiment with different features
- Build trading strategies using predictions
- Track performance over time